# Basic book-agent evaluation

The curated dataset currently contains ten English Pet Sematary questions. Each case includes `book_refs`: the exact EPUB SHA-256 references resolved to current catalog IDs before invoking the full agent. This notebook generates answers for manual inspection; use `uv run python eval.py` for DeepEval scoring.

Use the project `.venv` kernel (`uv sync --group notebooks`). Running the eval requires `.env` with `DATABASE_URL` and `OPENAI_API_KEY`, and the referenced EPUB already ingested. It makes paid model/embedding calls. Each question is stateless; results overwrite `evals/basic_result.json` and are saved after each case. Notebook outputs and result files can contain book passages: clear outputs before committing.

In [ ]:
import json
from html import escape
from pathlib import Path

from IPython.display import HTML, display

from rag_library.evaluation import run_evaluation

ROOT = Path.cwd().resolve()
if not (ROOT / "evals/basic.json").exists():
    ROOT = ROOT.parent
assert (ROOT / "evals/basic.json").exists(), "Start Jupyter in the repo or notebooks folder"
DATASET = ROOT / "evals/basic.json"
OUTPUT = ROOT / "evals/basic_result.json"

## Preview the question set

In [ ]:
cases = json.loads(DATASET.read_text(encoding="utf-8"))
print(f"{len(cases)} questions")
cases[:2]

## Run the agent

Run this cell to make live calls. To review a previous run without making calls, skip it and continue below. A failed question is recorded with an error and the remaining cases continue.

In [ ]:
results = run_evaluation(DATASET, OUTPUT)
print(f"Saved {len(results)} results to {OUTPUT}")

## Compare answers

Read the answers yourself. Different wording can still be correct. A missing or incorrect answer is a starting point for inspecting retrieval and tool use.

In [ ]:
results = json.loads(OUTPUT.read_text(encoding="utf-8"))
headers = ["ID", "Question", "Expected", "Actual", "Error"]
keys = ["id", "question", "expected_answer", "actual_answer", "error"]
rows = "".join(
    "<tr>" + "".join(
        f"<td style='vertical-align:top;text-align:left'>{escape(str(row.get(key) or ''))}</td>"
        for key in keys
    ) + "</tr>"
    for row in results
)
display(HTML("<table><thead><tr>" + "".join(f"<th>{h}</th>" for h in headers) + "</tr></thead><tbody>" + rows + "</tbody></table>"))

## Inspect one answer and its sources

Change `case_id` to inspect the actual tool calls, RAG answer, and retrieved book/chapter/chunk references.

In [ ]:
case_id = "cementerio-01"
result = next(row for row in results if row["id"] == case_id)
print(json.dumps(result, ensure_ascii=False, indent=2))